# BLUE IA — Experimento 001 🔵
**Blue Lab Studios | Primer laboratorio de inteligencia artificial**

En este cuaderno conocerás un modelo real de pesos abiertos: `Qwen/Qwen2.5-0.5B-Instruct`. Se descarga desde Hugging Face y se ejecuta en el entorno de Colab, sin API comercial ni claves. La personalidad de Blue se expresa mediante instrucciones de texto: **no se entrenan ni se modifican los parámetros**.

Ejecuta las celdas en orden con ▶. La primera descarga necesita conexión y puede tardar varios minutos. Si hay GPU, se usará automáticamente; en CPU también funciona, aunque más lentamente. Las respuestas pueden contener errores: comprueba los datos importantes.

## 1. Preparar las herramientas
PyTorch ejecuta las operaciones del modelo; Transformers proporciona el modelo y su tokenizador. Colab normalmente ya incluye PyTorch: solo se instala si falta. La versión de Transformers se fija para que el ejemplo sea reproducible.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.48.3", "safetensors"
])
print("Dependencias preparadas. Continúa con la siguiente celda.")

## 2. Descargar y cargar el modelo
El **tokenizador** traduce texto a unidades numéricas llamadas tokens y vuelve a convertir tokens en texto. Los **parámetros** son los valores numéricos aprendidos durante el entrenamiento previo: determinan cómo el modelo calcula sus predicciones. En este experimento solo los leemos.

Los archivos se descargan automáticamente la primera vez y se guardan en la caché temporal del entorno. En GPU usamos precisión de 16 bits; en CPU, 32 bits. El modelo pesa aproximadamente 1 GB en 16 bits o 2 GB en 32 bits, sin contar memoria adicional y archivos de caché.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_ID = "Qwen/Qwen2.5-0.5B-Instruct"
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
precision = torch.float16 if dispositivo.type == "cuda" else torch.float32
print(f"Dispositivo: {dispositivo}; precisión: {precision}")
if dispositivo.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Se usará CPU; la generación puede tardar bastante.")

# Se descargan exclusivamente los archivos necesarios de este modelo y su tokenizador.
tokenizador = AutoTokenizer.from_pretrained(MODELO_ID)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID, torch_dtype=precision, use_safetensors=True
).to(dispositivo)
modelo.eval()  # Modo de evaluación; no entrena ni cambia los pesos.
print(f"Modelo listo. Parámetros: {sum(p.numel() for p in modelo.parameters()):,}")

## 3. La personalidad y la memoria de Blue
Las instrucciones iniciales describen a Blue como una pequeña inteligencia de energía azul. Son parte del contexto, no una modificación del modelo. La memoria solo existe durante la sesión: conserva hasta tres intercambios recientes y tiene un límite total de tokens. Cuando el contexto no cabe, se retiran primero los intercambios más antiguos.

In [ ]:
INSTRUCCIONES_BLUE = (
    "Eres Blue, una pequeña inteligencia de energía azul de Blue Lab Studios. "
    "Eres amigable, curiosa y dedicada al conocimiento. Responde en español "
    "con claridad y de forma concisa. Si no sabes algo, dilo con honestidad."
)
MAX_INTERCAMBIOS = 3
MAX_TOKENS_ENTRADA = 512
MAX_TOKENS_NUEVOS = 128
historial = []  # Alterna mensajes del usuario y de Blue; no se guarda en archivos.


def reiniciar_conversacion():
    """Olvida los intercambios de esta sesión, conservando la personalidad de Blue."""
    historial.clear()


def hablar_con_blue(mensaje):
    """Genera una respuesta real y conserva una memoria breve de la conversación."""
    if not isinstance(mensaje, str) or not mensaje.strip():
        raise ValueError("Escribe un mensaje de texto que no esté vacío.")
    mensaje = mensaje.strip()
    anteriores = list(historial[-2 * MAX_INTERCAMBIOS:])
    sistema = {"role": "system", "content": INSTRUCCIONES_BLUE}
    usuario = {"role": "user", "content": mensaje}

    # La plantilla de chat añade los marcadores que este modelo espera.
    # Descartamos intercambios antiguos completos antes de generar.
    while True:
        mensajes = [sistema] + anteriores + [usuario]
        tokens = tokenizador.apply_chat_template(
            mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        )
        if tokens.shape[-1] <= MAX_TOKENS_ENTRADA:
            break
        if anteriores:
            anteriores = anteriores[2:]
        else:
            raise ValueError("El mensaje es demasiado largo; prueba con uno más corto.")

    tokens = tokens.to(dispositivo)
    # Inferencia: usar los parámetros ya aprendidos para predecir, sin entrenar.
    # Generación: se predice un token a la vez hasta terminar o llegar al límite.
    with torch.inference_mode():
        salida = modelo.generate(
            input_ids=tokens,
            attention_mask=torch.ones_like(tokens),
            max_new_tokens=MAX_TOKENS_NUEVOS,
            do_sample=False,
            pad_token_id=tokenizador.eos_token_id,
        )
    nuevos_tokens = salida[0, tokens.shape[-1]:]
    respuesta = tokenizador.decode(nuevos_tokens, skip_special_tokens=True).strip()
    # Solo guardamos la conversación si la generación terminó correctamente.
    historial[:] = (anteriores + [usuario, {"role": "assistant", "content": respuesta}])[-2 * MAX_INTERCAMBIOS:]
    return respuesta

print("Blue está preparada. Puedes conversar en la siguiente sección.")

## 4. Hablar con Blue
Edita el texto de esta celda y ejecútala tantas veces como quieras. Blue conserva una memoria corta mientras el entorno siga activo. Para empezar de nuevo, ejecuta `reiniciar_conversacion()` en otra celda. Si aparece un error por longitud, reduce el mensaje.

In [ ]:
print("🔵 Blue:", hablar_con_blue("Hola, Blue. ¿Quién eres?"))

### Conversación continua (opcional)
Ejecuta la celda, escribe un mensaje y pulsa Intro. Escribe `salir` para terminar. Si tu navegador móvil no muestra la entrada de texto correctamente, usa la celda anterior.

In [ ]:
while True:
    mensaje = input("Tú (escribe salir para terminar): ").strip()
    if mensaje.lower() == "salir":
        print("🔵 Blue: ¡Hasta pronto!")
        break
    try:
        print("🔵 Blue:", hablar_con_blue(mensaje))
    except ValueError as error:
        print("Aviso:", error)

## 5. Ejemplos para explorar
Ejecuta cada celda por separado. Para comparar respuestas sin contexto previo, ejecuta `reiniciar_conversacion()` antes de cada pregunta. Las respuestas son generadas por el modelo y pueden variar o contener errores.

In [ ]:
print("🔬 Ciencia:", hablar_con_blue("Explica qué es la fotosíntesis en tres frases sencillas."))

In [ ]:
print("➗ Matemáticas:", hablar_con_blue("¿Cuánto es 12 por 7? Explica cómo calcularlo."))

In [ ]:
print("💻 Tecnología:", hablar_con_blue("¿Qué diferencia hay entre hardware y software? Da un ejemplo de cada uno."))

## 6. Pruebas básicas
Esta celda realiza una inferencia real, comprueba que devuelve texto y verifica el límite de la memoria y la validación de entrada. Borrará el historial antes y después. Las pruebas no garantizan que las respuestas sean correctas ni evalúan su calidad.

In [ ]:
reiniciar_conversacion()
try:
    respuesta_prueba = hablar_con_blue("Saluda brevemente en español.")
    assert isinstance(respuesta_prueba, str) and respuesta_prueba.strip(), "La respuesta está vacía."
    assert len(historial) == 2, "El intercambio no se guardó correctamente."
    assert len(historial) <= 2 * MAX_INTERCAMBIOS, "El historial supera el límite."
    try:
        hablar_con_blue("   ")
    except ValueError:
        pass
    else:
        raise AssertionError("Se esperaba rechazar un mensaje vacío.")
    assert len(historial) == 2, "Un mensaje inválido cambió el historial."
    print("Pruebas básicas aprobadas. Respuesta real:", respuesta_prueba)
finally:
    reiniciar_conversacion()

## ¿Qué acabamos de hacer?
1. El tokenizador convirtió la conversación en números con una plantilla de chat.
2. El modelo aplicó sus parámetros existentes para predecir el siguiente token repetidamente.
3. El tokenizador convirtió los tokens nuevos en texto. Solo se guarda una ventana breve de la conversación.

Este es un primer paso de **inferencia**, no la creación ni el entrenamiento de un modelo propio. En futuros experimentos podrás estudiar datos, entrenamiento y evaluación. Si el entorno se reinicia, vuelve a ejecutar las celdas desde el principio.

# Experimento 002 — Interfaz web de Blue IA 💙

Conversaremos con el **mismo modelo y tokenizador ya cargados** en este entorno. Esta sección no vuelve a descargar ni a entrenar el modelo. En una sesión nueva, ejecuta primero las celdas **1. Preparar las herramientas** y **2. Descargar y cargar el modelo**; no hace falta ejecutar la conversación de consola. Después ejecuta las celdas de esta sección en orden.

**El enlace temporal de Gradio es público para cualquiera que lo conozca.** No escribas información privada ni compartas el enlace con personas a las que no quieras dar acceso. La conversación se mantiene solo en la sesión web y no es aprendizaje permanente. El enlace dejará de funcionar al detener el entorno de Colab o cuando expire el túnel de Gradio. La interfaz usa el servicio de túnel de Gradio para compartir el acceso; las respuestas las genera Qwen dentro de Colab, sin API comercial de modelos.

### 1. Instalar Gradio
Esta versión fija evita incompatibilidades con cambios futuros de la interfaz. Si Colab solicita reiniciar el entorno después de la instalación, reinícialo y vuelve a cargar el modelo antes de continuar.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio==5.49.1"])
import gradio as gr
if gr.__version__ != "5.49.1":
    raise RuntimeError(
        "Colab tiene otra versión de Gradio cargada. Reinicia el entorno y vuelve a cargar el modelo."
    )
print("Gradio instalado. Continúa con la siguiente celda.")

### 2. Conectar Blue y preparar la memoria
Cada sesión web tiene su propio historial limitado. La interfaz y la consola tienen memorias independientes. Si el contexto supera el límite de tokens, se retiran primero los intercambios más antiguos completos. Las solicitudes al modelo se procesan de una en una.

In [ ]:
import threading
import torch
import gradio as gr

if "modelo" not in globals() or "tokenizador" not in globals():
    raise RuntimeError("Primero ejecuta la celda 2: Descargar y cargar el modelo.")

# Si la sección 3 de la consola no se ejecutó, usamos las mismas instrucciones.
PERSONALIDAD_WEB = globals().get("INSTRUCCIONES_BLUE", (
    "Eres Blue, una pequeña inteligencia de energía azul de Blue Lab Studios. "
    "Eres amigable, curiosa y dedicada al conocimiento. Responde en español "
    "con claridad y de forma concisa. Si no sabes algo, dilo con honestidad."
))
MAX_INTERCAMBIOS_WEB = 3
MAX_TOKENS_ENTRADA_WEB = 512
MAX_TOKENS_NUEVOS_WEB = 128
# Gradio también limita la cola; el bloqueo protege el modelo entre callbacks.
if "_bloqueo_blue_web" not in globals():
    _bloqueo_blue_web = threading.Lock()


def generar_respuesta_web(mensaje, memoria):
    """Usa el modelo existente y devuelve respuesta y memoria nuevas, sin cambiar la entrada."""
    if not isinstance(mensaje, str) or not mensaje.strip():
        raise ValueError("Escribe un mensaje antes de enviarlo.")
    mensaje = mensaje.strip()
    anteriores = list((memoria or [])[-2 * MAX_INTERCAMBIOS_WEB:])
    usuario = {"role": "user", "content": mensaje}
    sistema = {"role": "system", "content": PERSONALIDAD_WEB}
    with _bloqueo_blue_web:
        while True:
            tokens = tokenizador.apply_chat_template(
                [sistema] + anteriores + [usuario], tokenize=True,
                add_generation_prompt=True, return_tensors="pt"
            )
            if tokens.shape[-1] <= MAX_TOKENS_ENTRADA_WEB:
                break
            if anteriores:
                anteriores = anteriores[2:]
            else:
                raise ValueError("El mensaje es demasiado largo; prueba con uno más corto.")
        # El modelo ya decide dónde está: CPU o GPU. No se cargan pesos nuevos.
        dispositivo_modelo = next(modelo.parameters()).device
        tokens = tokens.to(dispositivo_modelo)
        with torch.inference_mode():
            salida = modelo.generate(
                input_ids=tokens, attention_mask=torch.ones_like(tokens),
                max_new_tokens=MAX_TOKENS_NUEVOS_WEB, do_sample=False,
                temperature=None, top_p=None, top_k=None,
                pad_token_id=tokenizador.eos_token_id,
            )
        respuesta = tokenizador.decode(
            salida[0, tokens.shape[-1]:], skip_special_tokens=True
        ).strip()
    if not respuesta:
        raise RuntimeError("El modelo no devolvió texto.")
    nueva_memoria = (anteriores + [usuario, {"role": "assistant", "content": respuesta}])[-2 * MAX_INTERCAMBIOS_WEB:]
    return respuesta, nueva_memoria


def responder_web(mensaje, memoria):
    """Actualiza solo la sesión activa; no guarda mensajes en archivos."""
    memoria = list(memoria or [])
    if not isinstance(mensaje, str) or not mensaje.strip():
        yield memoria, memoria, gr.update(), gr.update(), gr.update(), "Escribe un mensaje antes de enviarlo."
        return
    provisional = memoria + [{"role": "user", "content": mensaje.strip()}]
    # Mostrar el mensaje y el indicador antes de iniciar la inferencia.
    yield provisional, memoria, gr.update(value="", interactive=False), gr.update(interactive=False), gr.update(interactive=False), "Blue está pensando…"
    try:
        _, actualizada = generar_respuesta_web(mensaje, memoria)
        yield actualizada, actualizada, gr.update(value="", interactive=True), gr.update(interactive=True), gr.update(interactive=True), ""
    except ValueError as error:
        yield memoria, memoria, gr.update(value=mensaje, interactive=True), gr.update(interactive=True), gr.update(interactive=True), str(error)
    except Exception:
        # Evitar mostrar detalles internos o conversaciones en la salida del servidor.
        yield memoria, memoria, gr.update(value=mensaje, interactive=True), gr.update(interactive=True), gr.update(interactive=True), "No pude generar una respuesta. Revisa que el modelo siga cargado y vuelve a intentarlo."


def nueva_conversacion_web():
    return [], [], "", ""

print("Interfaz preparada para usar el modelo en:", next(modelo.parameters()).device)

### 3. Construir la interfaz
El cuadro de texto admite Enter y el botón **Enviar**. **Nueva conversación** borra solo la conversación de esa sesión. Cada respuesta utiliza como máximo tres intercambios y 512 tokens de entrada; se generan hasta 128 tokens nuevos.

In [ ]:
# Al reconstruir la interfaz, cerramos la anterior si estaba abierta.
if "interfaz_blue" in globals():
    interfaz_blue.close()

estilo_blue = """
body, .gradio-container { background: #07101f !important; color: #e8f3ff !important; }
.gradio-container { max-width: 760px !important; margin: auto !important; }
#titulo-blue { text-align: center; padding: 8px 0; }
#titulo-blue h1 { color: #7dd3fc; margin-bottom: 0; }
#titulo-blue p { color: #a5bdd5; }
#chat-blue { border: 1px solid #1d4e78; border-radius: 18px; }
#estado-blue { color: #7dd3fc; min-height: 1.5em; }
@media (max-width: 600px) { .gradio-container { padding: 8px !important; } }
"""
with gr.Blocks(
    title="Blue IA | Blue Lab Studios", css=estilo_blue,
    theme=gr.themes.Base(primary_hue="blue", neutral_hue="slate", font=["system-ui", "sans-serif"]),
    analytics_enabled=False,
    js="() => { document.body.classList.add('dark'); }",
) as interfaz_blue:
    gr.Markdown("# 🔵 Blue IA\nBlue Lab Studios · Explora, pregunta y aprende", elem_id="titulo-blue")
    chat = gr.Chatbot(label="Conversación", type="messages", height=420, elem_id="chat-blue")
    memoria_sesion = gr.State([])
    estado = gr.Markdown("", elem_id="estado-blue")
    mensaje_web = gr.Textbox(label="Tu mensaje", placeholder="Escribe aquí tu pregunta…", lines=1, max_lines=4)
    with gr.Row():
        enviar = gr.Button("Enviar", variant="primary")
        nueva = gr.Button("Nueva conversación")
    entradas = [mensaje_web, memoria_sesion]
    salidas = [chat, memoria_sesion, mensaje_web, enviar, nueva, estado]
    enviar.click(responder_web, entradas, salidas, api_name=False,
                 concurrency_id="modelo_blue", concurrency_limit=1, trigger_mode="once")
    mensaje_web.submit(responder_web, entradas, salidas, api_name=False,
                       concurrency_id="modelo_blue", concurrency_limit=1, trigger_mode="once")
    nueva.click(nueva_conversacion_web, None, [chat, memoria_sesion, mensaje_web, estado],
                api_name=False, concurrency_id="modelo_blue", concurrency_limit=1)

interfaz_blue.queue(default_concurrency_limit=1)
print("Interfaz construida. Ejecuta la siguiente celda para abrirla.")

### 4. Abrir el enlace en tu celular
Ejecuta la siguiente celda y espera a que aparezca **Enlace temporal**. Tócalo o cópialo en Chrome para Android. Solo compártelo con personas de confianza: **cualquier persona con el enlace puede acceder**. Gradio lo desactivará cuando pare Colab o expire el túnel. No es necesario dejar abierta una consola de conversación.

In [ ]:
_, _, enlace_temporal = interfaz_blue.launch(
    share=True, prevent_thread_lock=True, show_error=False, inbrowser=False
)
if enlace_temporal:
    print("\n🔵 Enlace temporal de Blue IA:", enlace_temporal)
    print("Cualquier persona con este enlace puede acceder. Compártelo solo si lo deseas.")
else:
    print("No se pudo crear el enlace temporal. Revisa la salida de Gradio y vuelve a intentarlo.")

### 5. Detener o reiniciar la interfaz
Para detener el enlace sin cerrar Colab, ejecuta la siguiente celda. Para reiniciar, vuelve a ejecutar **3. Construir la interfaz** y **4. Abrir el enlace**; el modelo sigue cargado mientras el entorno no se reinicie. Si detienes o reinicias el entorno de Colab, tendrás que volver a cargarlo.

In [ ]:
interfaz_blue.close()
print("Interfaz detenida. El enlace temporal ya no está disponible.")